# S5 — EDA 大統整：從零到報告

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`../datasets/spaceship-titanic/train.csv`（全新完整 EDA）  
> **學完能幹嘛**：面對任何 Kaggle 資料集，用系統化的 EDA 流程產出完整分析報告

---

## 這堂課的目標

S1-S4 學了一堆「招式」，這堂課把招式串成「套路」。

我們會對 **Spaceship Titanic** 做一次從頭到尾的 EDA，按照固定的 checklist 走：

```
Step 1: First Look     (S1 技巧)
Step 2: Missing Data   (S2 技巧)
Step 3: Distributions  (S3 技巧)
Step 4: Features       (S4 技巧)
Step 5: Correlations   (本堂新增)
Step 6: Summary        (整合報告)
```

**一句話口訣：比賽前 80% 的時間在看資料，只有 20% 在調模型 — 這堂課就是那 80% 的完整流程。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('../datasets/spaceship-titanic/train.csv')
print(f'Spaceship Titanic: {df.shape[0]} rows × {df.shape[1]} columns')

---
## Step 1: First Look (S1 回顧)

In [ ]:
# 三連發
print(f'Shape: {df.shape}')
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
# Target 分析
print('=== Target: Transported ===')
print(df['Transported'].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(5, 5))
df['Transported'].value_counts().plot.pie(
    autopct='%1.1f%%', colors=['#ff9999', '#66b2ff'],
    labels=['Not Transported', 'Transported'], ax=ax
)
ax.set_title('Transported 比例', fontsize=14)
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('→ 接近 50:50，不需要擔心類別不平衡')

In [ ]:
# 欄位分類
num_cols = df.select_dtypes(include='number').columns.tolist()
obj_cols = df.select_dtypes(include='object').columns.tolist()
bool_cols = df.select_dtypes(include='bool').columns.tolist()

print(f'數值型: {num_cols}')
print(f'物件型: {obj_cols}')
print(f'布林型: {bool_cols}')
print(f'\n各欄位唯一值：')
print(df.nunique().sort_values())

---
## Step 2: Missing Data (S2 回顧)

In [ ]:
# NA 全景圖
na_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]

fig, ax = plt.subplots(figsize=(10, 5))
na_pct.plot(kind='bar', color='coral', edgecolor='white', ax=ax)
ax.set_title('Spaceship Titanic — 缺值比例 (%)', fontsize=14)
ax.set_ylabel('%')
for i, v in enumerate(na_pct):
    ax.text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'有缺值的欄位：{len(na_pct)} 個')
print('→ 所有欄位缺值率都在 2-3%，屬於低缺值率')

In [ ]:
# 缺值處理策略
df_clean = df.copy()
df_clean['Transported_int'] = df_clean['Transported'].astype(float)  # 含 NaN 安全轉換

# 數值欄位：中位數填補
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols + ['Age']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 類別欄位：眾數填補
for col in ['HomePlanet', 'Destination', 'CryoSleep', 'VIP']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('填補後缺值（不含 Cabin 和 Name）：')
remaining = df_clean.drop(columns=['Cabin', 'Name']).isnull().sum()
print(remaining[remaining > 0] if remaining.any() else '✅ 無缺值')

---
## Step 3: Distributions (S3 回顧)

In [ ]:
# 數值型分布
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
num_features = ['Age'] + spend_cols

for i, col in enumerate(num_features):
    ax = axes[i // 3, i % 3]
    for val in [True, False]:
        subset = df_clean[df_clean['Transported'] == val][col].dropna()
        sns.kdeplot(subset, fill=True, ax=ax, label=f'Transported={val}', alpha=0.5)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('數值型特徵 — KDE by Transported', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 類別型分布
cat_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, col in enumerate(cat_features):
    ax = axes[i // 2, i % 2]
    rate = df_clean.groupby(col)['Transported_int'].mean().sort_values(ascending=False)
    rate.plot(kind='bar', color='steelblue', edgecolor='white', ax=ax)
    ax.set_title(f'Transported Rate by {col}', fontsize=11)
    ax.set_ylabel('Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.suptitle('類別型特徵 — Transported Rate', fontsize=14)
plt.tight_layout()
plt.show()

print('關鍵發現：')
print('  - CryoSleep=True 的 transported rate 極高 (~82%)')
print('  - HomePlanet=Europa 的 transported rate 較高 (~66%)')
print('  - VIP 對 transported 影響不大')

---
## Step 4: Feature Engineering (S4 回顧)

In [ ]:
# 特徵 1：TotalSpending
df_clean['TotalSpending'] = df_clean[spend_cols].sum(axis=1)

# 特徵 2：Cabin 拆解
cabin_split = df_clean['Cabin'].str.split('/', expand=True)
df_clean['Cabin_deck'] = cabin_split[0]
df_clean['Cabin_num'] = pd.to_numeric(cabin_split[1], errors='coerce')
df_clean['Cabin_side'] = cabin_split[2]

# 特徵 3：是否消費為零
df_clean['NoSpending'] = (df_clean['TotalSpending'] == 0).astype(int)

# 特徵 4：Group（從 PassengerId 拆出）
df_clean['Group'] = df_clean['PassengerId'].str.split('_').str[0]
df_clean['GroupSize'] = df_clean.groupby('Group')['Group'].transform('count')
df_clean['IsAlone'] = (df_clean['GroupSize'] == 1).astype(int)

print('新特徵一覽：')
new_feats = ['TotalSpending', 'Cabin_deck', 'Cabin_side', 'NoSpending', 'GroupSize', 'IsAlone']
for feat in new_feats:
    print(f'  {feat}: nunique={df_clean[feat].nunique()}, na={df_clean[feat].isnull().sum()}')

---
## 核心觀念 1／3 — 相關性分析：找出最重要的變數

到這一步，你已經有了原始 + 新建的特徵。現在要排優先級：**哪些特徵最可能有用？**

In [ ]:
# 數值特徵 vs Target 相關性
numeric_df = df_clean.select_dtypes(include='number')
corr_with_target = numeric_df.corr()['Transported_int'].drop('Transported_int').sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
corr_with_target.plot(kind='barh', color=corr_with_target.apply(
    lambda x: '#ff6b6b' if x < 0 else '#4ecdc4'
), ax=ax)
ax.set_title('各特徵與 Transported 的相關性', fontsize=14)
ax.set_xlabel('Correlation')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('Top 正相關：', corr_with_target.tail(3).index.tolist())
print('Top 負相關：', corr_with_target.head(3).index.tolist())

In [ ]:
# 相關性矩陣熱力圖（數值特徵之間）
key_nums = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa',
            'VRDeck', 'TotalSpending', 'GroupSize', 'Transported_int']
corr_matrix = df_clean[key_nums].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # 只顯示下三角
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax)
ax.set_title('特徵相關性矩陣', fontsize=14)
plt.tight_layout()
plt.show()

print('觀察：')
print('  - Spa 和 VRDeck 高度正相關 (0.XX) — 可能有共線性')
print('  - TotalSpending 跟各消費欄位高度相關（因為是加總）')

In [ ]:
# 類別特徵 vs Target — 用 groupby mean 模擬「相關性」
cat_analysis = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP',
                'Cabin_deck', 'Cabin_side', 'NoSpending', 'IsAlone']

cat_impact = {}
for col in cat_analysis:
    rates = df_clean.groupby(col)['Transported_int'].mean()
    cat_impact[col] = rates.std()  # 標準差越大，區分度越高

impact_series = pd.Series(cat_impact).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
impact_series.plot(kind='bar', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('類別特徵的 Target 區分度（rate 標準差）', fontsize=14)
ax.set_ylabel('Std of Transported Rate')
plt.tight_layout()
plt.show()

print('→ CryoSleep 和 NoSpending 的區分度最高 — 這是最重要的特徵')

**口訣**：數值用 `corr()` 排序，類別用 `groupby mean` 的 std 排序 — 找出最有區分度的特徵優先處理。

---
## 核心觀念 2／3 — EDA Checklist 模板

把 S1-S5 的技巧整理成一個**可重用的 checklist**：

| Step | 做什麼 | 工具 | 輸出 |
|:-----|:-------|:-----|:-----|
| 1. First Look | shape/info/describe/target | S1 三連發 | 欄位分類地圖 |
| 2. Missing | NA bar/heatmap/mechanism | S2 策略 | 填補策略表 |
| 3. Distribution | hist/KDE/boxplot/crosstab | S3 三板斧 | 關鍵發現列表 |
| 4. Features | combine/extract/bin | S4 技巧 | 新特徵列表 |
| 5. Correlation | corr heatmap/target ranking | 本堂 | 優先特徵排序 |
| 6. Summary | 多面板 dashboard | 整合 | 分析報告 |

---
## 核心觀念 3／3 — 從 EDA 到決策

EDA 不是為了畫圖而畫圖，每個發現都應該對應一個**行動決策**。

In [ ]:
# 建立 EDA 發現 → 決策 對照表
findings = pd.DataFrame({
    '發現': [
        'CryoSleep=True 的 transported rate 極高 (~82%)',
        '消費為 0 的乘客 transported rate 高',
        'HomePlanet=Europa transported rate 較高',
        'Cabin_deck B, C 的 rate 較高',
        '消費欄位嚴重右偏',
        'GroupSize=1 和 >4 的 rate 較低',
        'VIP 對 target 影響很小',
    ],
    '決策': [
        '→ CryoSleep 是最強特徵，務必保留',
        '→ 建立 NoSpending 二元特徵',
        '→ HomePlanet 做 one-hot encoding',
        '→ Cabin_deck 做 one-hot 或 target encoding',
        '→ 對消費欄位做 log1p 轉換',
        '→ GroupSize 可能需要分箱（1, 2-4, 5+）',
        '→ VIP 可以考慮丟棄或降低優先級',
    ]
})
print(findings.to_string(index=False))

---
## Step 6: EDA Summary Dashboard

In [ ]:
# 6 面板 EDA 總結 Dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Target 分布
df_clean['Transported'].value_counts().plot.pie(
    autopct='%1.1f%%', colors=['#ff9999', '#66b2ff'], ax=axes[0, 0])
axes[0, 0].set_title('Target: Transported')
axes[0, 0].set_ylabel('')

# 2. 缺值概覽
na_pct_all = (df.isnull().mean() * 100)
na_pct_show = na_pct_all[na_pct_all > 0].sort_values(ascending=True)
na_pct_show.plot(kind='barh', color='coral', ax=axes[0, 1])
axes[0, 1].set_title('Missing Values (%)')

# 3. CryoSleep — 最強特徵
df_clean.groupby('CryoSleep')['Transported_int'].mean().plot(
    kind='bar', color=['#ff9999', '#66b2ff'], ax=axes[0, 2])
axes[0, 2].set_title('Transported by CryoSleep')
axes[0, 2].set_xticklabels(['False', 'True'], rotation=0)

# 4. TotalSpending 分布
for val in [True, False]:
    subset = df_clean[df_clean['Transported'] == val]['TotalSpending']
    sns.kdeplot(np.log1p(subset), fill=True, ax=axes[1, 0],
                label=f'Transported={val}', alpha=0.5)
axes[1, 0].set_title('log(TotalSpending) by Transported')
axes[1, 0].legend(fontsize=8)

# 5. HomePlanet
df_clean.groupby('HomePlanet')['Transported_int'].mean().sort_values().plot(
    kind='bar', color='steelblue', ax=axes[1, 1])
axes[1, 1].set_title('Transported by HomePlanet')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

# 6. 特徵重要性
abs_corr = corr_with_target.abs().sort_values(ascending=True).tail(8)
abs_corr.plot(kind='barh', color='steelblue', ax=axes[1, 2])
axes[1, 2].set_title('Top 8 Feature Correlations')

plt.suptitle('Spaceship Titanic — EDA Summary Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 課堂練習（40 min）

### 🟢 送分題

用 Spaceship Titanic 資料集，計算並畫出**數值特徵的相關性矩陣**熱力圖。
- 只取以下欄位：`Age, RoomService, FoodCourt, ShoppingMall, Spa, VRDeck`
- 加上 `annot=True` 顯示數字

In [ ]:
# TODO: 你的程式碼

### 🟡 核心題

建立一個 **2×3 的子圖 dashboard**，包含：
1. Transported 圓餅圖
2. Age KDE by Transported
3. CryoSleep vs Transported bar chart
4. HomePlanet vs Transported bar chart
5. Cabin_deck vs Transported bar chart
6. TotalSpending boxplot by Transported

In [ ]:
# TODO: 你的程式碼

### 🔴 挑戰題（Take-home Portfolio）

對 **House Prices** 資料集，從零開始做一份完整 EDA 報告：

1. First Look：shape/info/describe + SalePrice 分布
2. Missing Data：找出缺值欄位，決定每個的處理策略
3. Distribution：找出前 5 個最右偏的欄位，做 log 轉換
4. Feature Engineering：至少建立 3 個新特徵
5. Correlation：畫出與 SalePrice 相關性前 10 名的特徵
6. Summary：建立 EDA dashboard + 發現→決策對照表

這份報告可以當作你的 **數據分析作品集** 的一部分。

In [ ]:
# TODO: 你的程式碼（建議新開一個 notebook）

---
## 小結與速查表

### EDA Checklist（可直接複製到任何新專案）

```
□ Step 1: First Look
  □ df.shape / df.info() / df.describe(include='all')
  □ Target: value_counts() 或 histogram
  □ 欄位分類地圖: select_dtypes + nunique

□ Step 2: Missing Data
  □ NA bar chart + heatmap
  □ 判斷機制: MCAR / MAR / MNAR
  □ 填補策略: 刪除 / 眾數 / 分組中位數 / 缺值旗標

□ Step 3: Distributions
  □ 數值型: histogram + KDE by target + skew
  □ 類別型: stacked bar + barplot(hue) + crosstab
  □ 離群值: scatter + IQR + clip

□ Step 4: Feature Engineering
  □ 組合特徵: 相加 / 二值化
  □ 文字拆解: str.extract / str.split / str[0]
  □ 分箱: qcut / cut
  □ 交互特徵: col_a + '_' + col_b

□ Step 5: Correlations
  □ 數值: corr() heatmap + target 排序
  □ 類別: groupby mean std 排序

□ Step 6: Summary
  □ Multi-panel dashboard
  □ 發現 → 決策 對照表
```

### 全課程速查

| Session | 核心口訣 |
|:--------|:---------|
| S1 | 拿到資料的前 10 分鐘，決定接下來 10 小時的方向 |
| S2 | 缺值有三種死法：MCAR、MAR、MNAR |
| S3 | 偏態是脾氣，離群值是秘密 |
| S4 | 原始欄位是食材，特徵工程是料理 |
| S5 | 80% 時間看資料，20% 調模型 |